<a href="https://colab.research.google.com/github/xeondesk/KothaGPT/blob/dev/ml/gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KothaGPT — GPU verification & smoke test

Inspect the local PyTorch/CUDA environment, list NVIDIA devices, confirm bf16/NCCL support, and run a deterministic one-step trainer smoke test on CUDA.

> Expected on CPU-only dev/CI machines: CUDA unavailable. Use `make train-smoke` for CPU fallback.

In [1]:
import torch
import torch.distributed as dist
from pathlib import Path
from tempfile import TemporaryDirectory

print("torch:", torch.__version__)
print("cuda:", getattr(torch.version, "cuda", None))


torch: 2.11.0+cpu
cuda: None


In [2]:
cuda = torch.cuda
available = bool(cuda.is_available())
count = int(cuda.device_count()) if available else 0
devices = [cuda.get_device_name(i) for i in range(count)]
bf16 = bool(cuda.is_bf16_supported()) if available else False

nccl = bool(dist.is_nccl_available())
gloo = bool(dist.is_gloo_available())

summary = {
    "torch_version": torch.__version__,
    "cuda_version": getattr(torch.version, "cuda", None),
    "cuda_available": available,
    "device_count": count,
    "devices": devices,
    "bf16_supported": bf16,
    "nccl_available": nccl,
    "gloo_available": gloo,
}
summary


{'torch_version': '2.11.0+cpu',
 'cuda_version': None,
 'cuda_available': False,
 'device_count': 0,
 'devices': [],
 'bf16_supported': False,
 'nccl_available': False,
 'gloo_available': True}

### Require CUDA

Raises if CUDA or NCCL is unavailable, mirroring `ml/gpu_verify.py`.

In [3]:
if not summary["cuda_available"]:
    raise RuntimeError(
        "CUDA is unavailable. Install a CUDA-enabled PyTorch build and a compatible NVIDIA driver."
    )
if not summary["nccl_available"]:
    raise RuntimeError(
        "CUDA is available but NCCL is unavailable. Install PyTorch with NCCL support."
    )
print("CUDA environment OK:", summary["devices"])

RuntimeError: CUDA is unavailable. Install a CUDA-enabled PyTorch build and a compatible NVIDIA driver.

In [ ]:
from ml.models import BaseModelConfig, KothaGPT, ModelConfig, TrainingConfig
from ml.trainer import CausalLMDataset, train

config = BaseModelConfig(
    model=ModelConfig(
        vocab_size=64,
        hidden_size=32,
        num_layers=1,
        num_heads=2,
        max_position_embeddings=16,
    ),
    training=TrainingConfig(batch_size=2, max_steps=1, mixed_precision="bf16"),
    data=type("DataConfig", (), {"tokenizer_path": "", "train": "", "validation": ""})(),
)

device = "cuda"
tokens = torch.randint(0, 64, (8, 16), device="cpu")
dataset = CausalLMDataset(tokens)

with TemporaryDirectory(prefix="kothagpt-gpu-smoke-") as directory:
    result = train(
        KothaGPT(config.model),
        config,
        dataset,
        None,
        out_dir=Path(directory),
        device=device,
    )

{"device": device, "step": result["step"], "tokens_per_sec": result.get("tokens_per_sec")}

### GPU vs CPU matmul micro-benchmark

Compare a dense matmul on CUDA vs CPU to confirm the GPU is being used.

In [ ]:
import time

size = 2048
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

def bench(fn, repeats=5):
    for _ in range(2):
        fn()
    times = []
    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return min(times)

t_cpu = bench(lambda: a_cpu @ b_cpu)

t_gpu = None
if torch.cuda.is_available():
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()
    t_gpu = bench(lambda: a_gpu @ b_gpu)

{"cpu_sec": t_cpu, "gpu_sec": t_gpu, "speedup": (t_cpu / t_gpu) if t_gpu else None}